# 🛢️ Oil Spill Detection in SAR Images — Semantic Segmentation

**Assignment 2 · U-Net (built from scratch) vs DeepLabV3+ (ImageNet-pretrained)**

| | |
|---|---|
| **Team** | _write the 4–5 member names here_ |
| **Dataset** | Krestenitis et al., *Oil Spill Identification from Satellite Images Using Deep Neural Networks*, Remote Sensing 2019, 11(15), 1762 — Kaggle: `nabilsherif/oil-spill` |
| **Task** | Classify every pixel of a SAR image into 5 classes: sea, oil spill, look-alike, ship, land |

### How to run this notebook on Kaggle
1. **Add data** → search `nabilsherif/oil-spill` and add it.
2. **Settings** → Accelerator **GPU T4** · **Internet ON** (needed for `pip install` and the ImageNet weights).
3. **Save Version → Save & Run All** so the run continues in the background. All outputs go to `/kaggle/working/outputs`.
4. When the run finishes, download `outputs/export.zip`. It holds the two best models and some test samples for the mobile app.

### Notebook map (follows the suggested report structure)
1. Setup & configuration
2. Dataset inspection: structure, sizes, exact RGB → class mapping
3. Preprocessing: label conversion, train/val split, class imbalance, normalisation
4. Data augmentation
5. Models: U-Net from scratch and DeepLabV3+ with pretrained weights
6. Loss functions
7. Training loop, metrics, checkpoints
8. Experiments: 2 × U-Net, 3 × DeepLabV3+
9. Training curves and overfitting analysis
10. Final evaluation on the untouched test set
11. Qualitative results and failure cases
12. Export for the mobile app

## 1. Setup & configuration

In [ ]:
# DeepLabV3+ comes from segmentation-models-pytorch (smp). We pin the version so that the
# saved weights load exactly the same way in the backend server used by the mobile app.
import subprocess, sys
from importlib.metadata import version, PackageNotFoundError

SMP_VERSION = "0.5.0"
try:
    installed = version("segmentation-models-pytorch")
except PackageNotFoundError:
    installed = None
if installed != SMP_VERSION:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"segmentation-models-pytorch=={SMP_VERSION}"], check=True)
print("segmentation-models-pytorch", SMP_VERSION, "ready")

In [ ]:
import os, json, time, random, shutil
from pathlib import Path
from collections import Counter

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp

# ------------------------------------------------------------------ configuration
SEED = 42
IMG_H, IMG_W = 320, 640      # training size. Originals are about 650 x 1250, so we keep the ~1:2 shape.
                             # Both sides must be divisible by 32 for U-Net and DeepLabV3+.
BATCH_SIZE = 8
EPOCHS = 40                  # same number of epochs for every experiment (fair comparison)
VAL_FRACTION = 0.10          # 10% of the official training set becomes our validation set
UNET_BASE = 32               # U-Net width: 32-64-128-256-512 channels
ENCODER = "resnet34"         # DeepLabV3+ backbone (ImageNet-pretrained)
FREEZE_EPOCHS = 5            # warm-up epochs with a frozen backbone (fine-tuning experiments)
MAX_IMAGES = None            # None = use every image

# QUICK_TEST=1 runs a tiny version of the whole notebook to check it end to end in a few minutes.
QUICK_TEST = os.environ.get("QUICK_TEST", "0") == "1"
if QUICK_TEST:
    IMG_H, IMG_W, BATCH_SIZE, EPOCHS, FREEZE_EPOCHS, MAX_IMAGES = 128, 256, 4, 2, 1, 24

NUM_CLASSES = 5
CLASS_NAMES = ["Sea surface", "Oil spill", "Look-alike", "Ship", "Land"]
CLASS_KEYS = ["sea", "oil", "lookalike", "ship", "land"]          # short names for tables
CLASS_COLORS = np.array([[0, 0, 0],        # 0 sea surface  - black
                         [0, 255, 255],    # 1 oil spill    - cyan
                         [255, 0, 0],      # 2 look-alike   - red
                         [153, 76, 0],     # 3 ship         - brown
                         [0, 153, 0]],     # 4 land         - green
                        dtype=np.uint8)
OIL, LOOKALIKE, SHIP = 1, 2, 3

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = DEVICE == "cuda"                 # mixed precision (faster) only on GPU
NUM_WORKERS = 2 if DEVICE == "cuda" else 0

OUT_DIR = Path("/kaggle/working/outputs") if Path("/kaggle/working").exists() else Path("outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

def seed_everything(seed=SEED):
    """Fix all random number generators so runs can be repeated."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

seed_everything()
# Reproducibility limitation: some GPU kernels (e.g. the backward pass of bilinear upsampling)
# are not deterministic, so two GPU runs can differ slightly even with the same seed.
torch.backends.cudnn.benchmark = False

LEGEND = [Patch(facecolor=CLASS_COLORS[i] / 255, edgecolor="gray", label=f"{i}: {n}") for i, n in enumerate(CLASS_NAMES)]
print(f"device={DEVICE} | torch {torch.__version__} | smp {smp.__version__} | quick_test={QUICK_TEST}")
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
print("outputs ->", OUT_DIR.resolve())

## 2. Dataset inspection

The Kaggle dataset has this layout (checked on the Kaggle data explorer):
```
oil-spill/
├── train/  images/ (1002 SAR images)   labels/ (1002 colour masks, img_XXXX.png)
└── test/   images/ (110 SAR images)    labels/ (110 colour masks)
```
The code below finds this folder automatically, so it works whatever the mount path is.

In [ ]:
def find_dataset_root():
    """Return the folder that contains train/images, train/labels, test/images and test/labels."""
    search = [Path(os.environ["DATA_ROOT"])] if "DATA_ROOT" in os.environ else []
    search += [Path("/kaggle/input"), Path("data")]
    for base in search:
        if not base.exists():
            continue
        for dirpath, _, _ in os.walk(base, followlinks=True):
            p = Path(dirpath)
            if all((p / split / sub).is_dir() for split in ["train", "test"] for sub in ["images", "labels"]):
                return p
    raise FileNotFoundError("Dataset not found. Add 'nabilsherif/oil-spill' to the notebook (or set DATA_ROOT).")

IMG_EXTS = {".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp"}

def list_pairs(split):
    """Match every image with its label file by file name (without extension)."""
    images = {p.stem: p for p in (DATA_ROOT / split / "images").iterdir() if p.suffix.lower() in IMG_EXTS}
    labels = {p.stem: p for p in (DATA_ROOT / split / "labels").iterdir() if p.suffix.lower() in IMG_EXTS}
    unmatched = sorted(images.keys() ^ labels.keys())
    if unmatched:
        print(f"⚠️ {split}: {len(unmatched)} files without a partner, e.g. {unmatched[:5]}")
    return [(images[k], labels[k]) for k in sorted(images.keys() & labels.keys())]

DATA_ROOT = find_dataset_root()
train_pairs_all = list_pairs("train")
test_pairs = list_pairs("test")
if MAX_IMAGES:
    train_pairs_all, test_pairs = train_pairs_all[:MAX_IMAGES], test_pairs[:max(4, MAX_IMAGES // 4)]

print("dataset root:", DATA_ROOT)
print(f"train pairs: {len(train_pairs_all)} | test pairs: {len(test_pairs)}")
print("image extensions:", Counter(p.suffix for p, _ in train_pairs_all + test_pairs))
print("label extensions:", Counter(p.suffix for _, p in train_pairs_all + test_pairs))
print("example:", train_pairs_all[0][0].name, "<->", train_pairs_all[0][1].name)

In [ ]:
# Image and mask sizes (PIL only reads the file header, so this is fast).
size_rows = []
for split, pairs in [("train", train_pairs_all), ("test", test_pairs)]:
    for ip, lp in pairs:
        im, lb = Image.open(ip), Image.open(lp)
        size_rows.append({"split": split, "image_size": im.size, "image_mode": im.mode,
                          "mask_size": lb.size, "mask_mode": lb.mode})
sizes = pd.DataFrame(size_rows)
print("image sizes (W, H):", dict(Counter(sizes.image_size)))
print("mask sizes  (W, H):", dict(Counter(sizes.mask_size)))
print("image modes:", dict(Counter(sizes.image_mode)), "| mask modes:", dict(Counter(sizes.mask_mode)))
mismatch = (sizes.image_size != sizes.mask_size).sum()
print("image/mask size mismatches:", mismatch)
assert mismatch == 0, "every mask must have the same size as its image"

### Verifying the exact RGB → class mapping
We count every distinct colour in **all** masks (train and test; we only read the labels, nothing is tuned on the
test set) and look up the nearest class colour from the assignment table. Distance 0 means an exact match.
A handful of anti-aliased border pixels with in-between colours are harmless: they get the nearest class.
Only a *frequent* colour far from every class would mean `CLASS_COLORS` is wrong.

In [ ]:
def color_table(mask_paths):
    counts = Counter()
    for p in mask_paths:
        m = cv2.imread(str(p), cv2.IMREAD_UNCHANGED)
        if m.ndim == 2:   # single-channel mask: treat each value as a grey colour
            m = np.stack([m] * 3, axis=-1)
        rgb = cv2.cvtColor(m[:, :, :3], cv2.COLOR_BGR2RGB).reshape(-1, 3).astype(np.int64)
        codes, n = np.unique(rgb[:, 0] * 65536 + rgb[:, 1] * 256 + rgb[:, 2], return_counts=True)
        counts.update(dict(zip(codes.tolist(), n.tolist())))
    rows = []
    total = sum(counts.values())
    for code, n in counts.most_common():
        rgb = np.array([code >> 16, (code >> 8) & 255, code & 255])
        dist = np.sqrt(((CLASS_COLORS.astype(int) - rgb) ** 2).sum(axis=1))
        rows.append({"rgb": tuple(int(v) for v in rgb), "pixels": n, "share_%": round(100 * n / total, 4),
                     "nearest_class": CLASS_NAMES[dist.argmin()], "distance": round(float(dist.min()), 1)})
    return pd.DataFrame(rows)

all_masks = [lp for _, lp in train_pairs_all + test_pairs]
colors = color_table(all_masks)
print(f"{len(colors)} distinct colours in {len(all_masks)} masks")
display(colors.head(10))
n_exact = colors.loc[colors.distance == 0, "pixels"].sum()
n_other = colors.loc[colors.distance > 0, "pixels"].sum()
print(f"pixels with an exact class colour: {n_exact:,} | other (border) pixels: {n_other:,} "
      f"({100 * n_other / (n_exact + n_other):.5f}%) -> mapped to the nearest class")
frequent_unknown = colors[(colors.distance > 40) & (colors["share_%"] > 0.01)]
if len(frequent_unknown):
    print("⚠️ frequent colours far from every class colour (check CLASS_COLORS!):")
    display(frequent_unknown)
else:
    print("✅ the 5 class colours from the assignment match the masks")

## 3. Preprocessing

**Label conversion.** Each RGB mask becomes a single-channel mask of class IDs 0–4. Each pixel gets the class
with the *nearest* palette colour, which also copes with any slightly-off colours at object borders.
Masks are only ever resized with **nearest-neighbour** interpolation and never normalised or colour-augmented,
so every label stays a valid class ID.

In [ ]:
def load_image(path):
    """Read an image as RGB uint8 (grey SAR images are repeated into 3 channels)."""
    img = cv2.imread(str(path), cv2.IMREAD_COLOR)
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

def rgb_mask_to_class(mask_rgb):
    """(H, W, 3) colour mask -> (H, W) class IDs, using the nearest class colour."""
    diff = mask_rgb[:, :, None, :].astype(np.int32) - CLASS_COLORS[None, None, :, :].astype(np.int32)
    return (diff ** 2).sum(axis=-1).argmin(axis=-1).astype(np.uint8)

def load_mask(path):
    m = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if m.ndim == 2:                       # already class IDs
        assert m.max() < NUM_CLASSES, f"{path.name}: unexpected label value {m.max()}"
        return m.astype(np.uint8)
    return rgb_mask_to_class(cv2.cvtColor(m[:, :, :3], cv2.COLOR_BGR2RGB))

def colorize(mask):
    """Class IDs -> RGB image for display."""
    return CLASS_COLORS[mask]

def overlay(image, mask, alpha=0.55):
    """Draw every non-sea class on top of the SAR image."""
    out = image.copy()
    m = mask > 0
    out[m] = (alpha * CLASS_COLORS[mask][m] + (1 - alpha) * image[m]).astype(np.uint8)
    return out

# Sanity check on one mask: the classes found and that colours -> IDs -> colours gives back the same mask.
m_rgb = load_image(train_pairs_all[0][1])
m_cls = load_mask(train_pairs_all[0][1])
print("class IDs in", train_pairs_all[0][1].name, "->", np.unique(m_cls).tolist())
print("round-trip pixel agreement:", f"{(colorize(m_cls) == m_rgb).all(axis=-1).mean() * 100:.3f}%")

**Train / validation split.** The official test set (110 images) stays untouched until Section 10.
We take a reproducible 10% of the official training set (seed 42) as the validation set.
Validation is used for model selection and early checks. The test set is never used for tuning.

In [ ]:
rng = np.random.default_rng(SEED)
perm = rng.permutation(len(train_pairs_all))
n_val = max(2, int(round(len(perm) * VAL_FRACTION)))
val_pairs = [train_pairs_all[i] for i in sorted(perm[:n_val])]
train_pairs = [train_pairs_all[i] for i in sorted(perm[n_val:])]
print(f"train {len(train_pairs)} | val {len(val_pairs)} | test {len(test_pairs)} (untouched)")

with open(OUT_DIR / "split.json", "w") as f:   # saved so the split can be reproduced exactly
    json.dump({"seed": SEED, "train": [p.name for p, _ in train_pairs], "val": [p.name for p, _ in val_pairs],
               "test": [p.name for p, _ in test_pairs]}, f, indent=1)

def load_split(pairs, name):
    """Load and resize all images/masks of a split into memory (fast training, no disk reads)."""
    images = np.zeros((len(pairs), IMG_H, IMG_W, 3), np.uint8)
    masks = np.zeros((len(pairs), IMG_H, IMG_W), np.uint8)
    t0 = time.time()
    for i, (ip, lp) in enumerate(pairs):
        images[i] = cv2.resize(load_image(ip), (IMG_W, IMG_H), interpolation=cv2.INTER_AREA)
        masks[i] = cv2.resize(load_mask(lp), (IMG_W, IMG_H), interpolation=cv2.INTER_NEAREST)
    print(f"{name}: {len(pairs)} pairs loaded in {time.time() - t0:.0f}s -> images {images.shape}, masks {masks.shape}")
    return images, masks

train_images, train_masks = load_split(train_pairs, "train")
val_images, val_masks = load_split(val_pairs, "val")

In [ ]:
# Look at several image/mask pairs. We pick one image for each rare class so all 5 labels show up.
def show_pairs(indices, images, masks, title):
    fig, axes = plt.subplots(len(indices), 3, figsize=(15, 3.2 * len(indices)))
    axes = np.atleast_2d(axes)
    for row, i in enumerate(indices):
        present = ", ".join(CLASS_NAMES[c] for c in np.unique(masks[i]))
        axes[row, 0].imshow(images[i]); axes[row, 0].set_title(f"SAR image #{i}")
        axes[row, 1].imshow(colorize(masks[i])); axes[row, 1].set_title(f"mask: {present}", fontsize=9)
        axes[row, 2].imshow(overlay(images[i], masks[i])); axes[row, 2].set_title("overlay (alignment check)")
        for ax in axes[row]:
            ax.axis("off")
    fig.legend(handles=LEGEND, loc="lower center", ncol=5)
    fig.suptitle(title)
    plt.tight_layout(rect=(0, 0.04, 1, 0.97))
    plt.show()

picked = []
for c in [OIL, LOOKALIKE, SHIP, 4]:
    has_c = np.where((train_masks == c).any(axis=(1, 2)))[0]
    if len(has_c):
        picked.append(int(has_c[0]))
picked += [int(i) for i in rng.choice(len(train_images), 2, replace=False)]
show_pairs(list(dict.fromkeys(picked)), train_images, train_masks, "Training image / mask pairs")

### Class imbalance
We count the pixels of each class and the number of images that contain each class.

In [ ]:
pixel_counts = np.bincount(train_masks.ravel(), minlength=NUM_CLASSES).astype(np.float64)
pixel_freq = pixel_counts / pixel_counts.sum()
image_counts = [(train_masks == c).any(axis=(1, 2)).sum() for c in range(NUM_CLASSES)]

imbalance = pd.DataFrame({"class": CLASS_NAMES, "pixel_share_%": (100 * pixel_freq).round(3),
                          "images_containing": image_counts})
display(imbalance)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].bar(CLASS_NAMES, 100 * pixel_freq, color=CLASS_COLORS / 255, edgecolor="gray")
axes[0].set_yscale("log"); axes[0].set_ylabel("share of pixels (%) — log scale"); axes[0].set_title("Pixel share per class")
axes[1].bar(CLASS_NAMES, image_counts, color=CLASS_COLORS / 255, edgecolor="gray")
axes[1].set_ylabel("number of training images"); axes[1].set_title("Images containing each class")
plt.tight_layout(); plt.savefig(OUT_DIR / "class_imbalance.png", dpi=120); plt.show()

**Class weights.** Some experiments use a weighted loss. We use *square-root median-frequency balancing*:
`w_c = sqrt(median(freq) / freq_c)`, normalised to a mean of 1. Rare classes (ships, oil) get larger weights.
The square root keeps the ship weight from becoming extreme, which would make training unstable.

In [ ]:
weights = np.sqrt(np.median(pixel_freq) / np.maximum(pixel_freq, 1e-8))
weights = weights / weights.mean()
CLASS_WEIGHTS = torch.tensor(weights, dtype=torch.float32)
display(pd.DataFrame({"class": CLASS_NAMES, "pixel_share_%": (100 * pixel_freq).round(3), "weight": weights.round(3)}))

**Normalisation.**
* **DeepLabV3+** uses an ImageNet-pretrained ResNet-34, so its inputs must be normalised **exactly like ImageNet**
  (mean = 0.485/0.456/0.406, std = 0.229/0.224/0.225). Otherwise the pretrained filters see inputs they were never trained on.
* **U-Net** is trained from scratch, so we use the **mean/std of our own training images**. This gives inputs
  with mean 0 and std 1 for this SAR data. The statistics come only from the training split, so the validation and test sets don't leak into them.

In [ ]:
IMAGENET_MEAN, IMAGENET_STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)

s, s2, n = np.zeros(3), np.zeros(3), 0
for img in train_images:                       # one image at a time to save memory
    x = img.reshape(-1, 3).astype(np.float64) / 255.0
    s += x.sum(axis=0); s2 += (x ** 2).sum(axis=0); n += len(x)
SAR_MEAN = tuple(float(v) for v in s / n)
SAR_STD = tuple(float(v) for v in np.sqrt(s2 / n - (s / n) ** 2))
print("SAR training-set mean:", np.round(SAR_MEAN, 4), "std:", np.round(SAR_STD, 4))

def norm_stats(model_type):
    return (IMAGENET_MEAN, IMAGENET_STD) if model_type == "deeplab" else (SAR_MEAN, SAR_STD)

## 4. Data augmentation
Augmentations are applied **only to the training set**.
* **Geometric** (applied to image **and** mask together so they stay aligned): horizontal flip, vertical flip
  (a satellite view has no fixed "up"), and random zoom (crop 60–100% of the scene and resize back).
* **Intensity** (applied to the **image only**): mild brightness/contrast change and multiplicative
  **speckle noise**, the kind of noise SAR images naturally have. Strong colour changes make no sense for single-band SAR, so we leave them out.

In [ ]:
def augment_pair(image, mask):
    if random.random() < 0.5:                                   # horizontal flip
        image, mask = image[:, ::-1], mask[:, ::-1]
    if random.random() < 0.5:                                   # vertical flip
        image, mask = image[::-1, :], mask[::-1, :]
    image, mask = np.ascontiguousarray(image), np.ascontiguousarray(mask)
    if random.random() < 0.5:                                   # random zoom
        h, w = mask.shape
        s = random.uniform(0.6, 1.0)
        ch, cw = int(h * s), int(w * s)
        y0, x0 = random.randint(0, h - ch), random.randint(0, w - cw)
        image = cv2.resize(image[y0:y0 + ch, x0:x0 + cw], (w, h), interpolation=cv2.INTER_LINEAR)
        mask = cv2.resize(mask[y0:y0 + ch, x0:x0 + cw], (w, h), interpolation=cv2.INTER_NEAREST)
    image = image.astype(np.float32)
    if random.random() < 0.5:                                   # brightness / contrast (image only)
        image = image * random.uniform(0.8, 1.2) + random.uniform(-20, 20)
    if random.random() < 0.3:                                   # speckle noise (image only)
        image = image * np.random.normal(1.0, 0.08, size=image.shape[:2])[:, :, None]
    return np.clip(image, 0, 255).astype(np.uint8), mask

def to_tensor(image, mean, std):
    """uint8 (H, W, 3) -> normalised float tensor (3, H, W)."""
    x = (image.astype(np.float32) / 255.0 - np.array(mean, np.float32)) / np.array(std, np.float32)
    return torch.from_numpy(x.transpose(2, 0, 1).copy())

class OilSpillDataset(Dataset):
    def __init__(self, images, masks, mean, std, augment=False):
        self.images, self.masks, self.mean, self.std, self.augment = images, masks, mean, std, augment

    def __len__(self):
        return len(self.images)

    def __getitem__(self, i):
        image, mask = self.images[i], self.masks[i]
        if self.augment:
            image, mask = augment_pair(image, mask)
        return to_tensor(image, self.mean, self.std), torch.from_numpy(mask.astype(np.int64))

def seed_worker(worker_id):
    """Give every DataLoader worker its own (reproducible) random seed."""
    worker_seed = torch.initial_seed() % 2 ** 32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

# Preview: the same image augmented 4 times. Mask colours must still sit on the same structures.
i = picked[0]
fig, axes = plt.subplots(2, 5, figsize=(20, 5))
axes[0, 0].imshow(train_images[i]); axes[0, 0].set_title("original")
axes[1, 0].imshow(overlay(train_images[i], train_masks[i]))
for k in range(1, 5):
    a_img, a_mask = augment_pair(train_images[i], train_masks[i])
    axes[0, k].imshow(a_img); axes[0, k].set_title(f"augmented #{k}")
    axes[1, k].imshow(overlay(a_img, a_mask))
for ax in axes.ravel():
    ax.axis("off")
plt.tight_layout(); plt.show()

## 5. Models

### Model A — U-Net from scratch
* **Encoder** (downsampling path): 4 blocks of *[3×3 conv → BatchNorm → ReLU] × 2*, each followed by 2×2 max-pooling.
  The resolution halves and the channels double: 32 → 64 → 128 → 256.
* **Bottleneck**: the deepest block (512 channels at 1/16 resolution). It has the largest receptive field and the most context.
* **Decoder** (upsampling path): 2×2 transposed convolutions double the resolution again.
* **Skip connections**: at each level the decoder **concatenates** the encoder feature map of the same size.
  Pooling throws away exact positions, and the skip connections bring that fine spatial detail back.
  This matters most for **thin oil slicks and tiny ships**, which would otherwise be blurred away.
* **Output layer**: a 1×1 convolution gives **5 logits per pixel** (one per class). `argmax` turns them into the predicted class mask.

No pretrained weights are used. Every weight starts random.

In [ ]:
class DoubleConv(nn.Module):
    """(3x3 conv -> BatchNorm -> ReLU) x 2"""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class UNet(nn.Module):
    def __init__(self, in_channels=3, num_classes=5, base=32):
        super().__init__()
        c1, c2, c3, c4, c5 = base, base * 2, base * 4, base * 8, base * 16
        # encoder
        self.enc1 = DoubleConv(in_channels, c1)
        self.enc2 = DoubleConv(c1, c2)
        self.enc3 = DoubleConv(c2, c3)
        self.enc4 = DoubleConv(c3, c4)
        self.pool = nn.MaxPool2d(2)
        # bottleneck
        self.bottleneck = DoubleConv(c4, c5)
        # decoder: upsample, concatenate the skip connection, then DoubleConv
        self.up4, self.dec4 = nn.ConvTranspose2d(c5, c4, 2, stride=2), DoubleConv(c4 * 2, c4)
        self.up3, self.dec3 = nn.ConvTranspose2d(c4, c3, 2, stride=2), DoubleConv(c3 * 2, c3)
        self.up2, self.dec2 = nn.ConvTranspose2d(c3, c2, 2, stride=2), DoubleConv(c2 * 2, c2)
        self.up1, self.dec1 = nn.ConvTranspose2d(c2, c1, 2, stride=2), DoubleConv(c1 * 2, c1)
        # 1x1 conv -> one score per class for every pixel
        self.head = nn.Conv2d(c1, num_classes, kernel_size=1)

    def forward(self, x):
        s1 = self.enc1(x)                     # full resolution
        s2 = self.enc2(self.pool(s1))         # 1/2
        s3 = self.enc3(self.pool(s2))         # 1/4
        s4 = self.enc4(self.pool(s3))         # 1/8
        b = self.bottleneck(self.pool(s4))    # 1/16
        d4 = self.dec4(torch.cat([self.up4(b), s4], dim=1))   # skip connection 4
        d3 = self.dec3(torch.cat([self.up3(d4), s3], dim=1))  # skip connection 3
        d2 = self.dec2(torch.cat([self.up2(d3), s2], dim=1))  # skip connection 2
        d1 = self.dec1(torch.cat([self.up1(d2), s1], dim=1))  # skip connection 1
        return self.head(d1)


def count_params(model):
    return sum(p.numel() for p in model.parameters())

# Shape check: the output must have 5 channels and the same height/width as the input.
_unet = UNet(num_classes=NUM_CLASSES, base=UNET_BASE)
_x = torch.randn(2, 3, IMG_H, IMG_W)
with torch.no_grad():
    _y = _unet(_x)
print(f"U-Net: input {tuple(_x.shape)} -> output {tuple(_y.shape)} | {count_params(_unet) / 1e6:.2f} M parameters")
assert _y.shape == (2, NUM_CLASSES, IMG_H, IMG_W)
print("predicted mask shape after argmax:", tuple(_y.argmax(1).shape))
del _unet, _x, _y

### Model B — DeepLabV3+ with pretrained weights
* **Architecture:** DeepLabV3+ from `segmentation-models-pytorch`, with a **ResNet-34 encoder pretrained on ImageNet**
  (`encoder_weights="imagenet"`). Atrous (dilated) convolutions in the **ASPP** module capture context at several scales.
  A light decoder fuses this context with low-level encoder features to recover sharp boundaries.
* **Head adapted to 5 classes:** `classes=5` gives a freshly initialised final conv layer with 5 outputs.
  Only the encoder is pretrained. The ASPP, decoder and head start from random weights.
* **Fine-tuning strategies** (compared in Section 8):
  *frozen backbone for the whole run* vs *frozen for the first `FREEZE_EPOCHS` epochs, then unfrozen with a 10× smaller learning rate for the encoder*.

The cell below **proves** the encoder really is pretrained by comparing it with a randomly initialised copy.

In [ ]:
def build_deeplab(pretrained=True):
    return smp.DeepLabV3Plus(encoder_name=ENCODER, encoder_weights="imagenet" if pretrained else None,
                             in_channels=3, classes=NUM_CLASSES)

_pre, _rand = build_deeplab(True), build_deeplab(False)
w_pre, w_rand = _pre.encoder.conv1.weight, _rand.encoder.conv1.weight
print(f"first conv layer: pretrained std={w_pre.std():.4f}, random-init std={w_rand.std():.4f}, "
      f"mean |difference|={(w_pre - w_rand).abs().mean():.4f}")
assert not torch.allclose(w_pre, w_rand), "encoder weights are NOT pretrained!"
print("✅ the ResNet-34 encoder is loaded with ImageNet weights")

with torch.no_grad():
    _y = _pre.eval()(torch.randn(2, 3, IMG_H, IMG_W))
print(f"DeepLabV3+: output {tuple(_y.shape)} | {count_params(_pre) / 1e6:.2f} M parameters "
      f"(encoder {count_params(_pre.encoder) / 1e6:.2f} M)")
assert _y.shape == (2, NUM_CLASSES, IMG_H, IMG_W)
del _pre, _rand, _y

## 6. Loss functions
| Loss | Idea | Why it may help | Why it may hurt |
|---|---|---|---|
| **Cross-Entropy (CE)** | average per-pixel log-loss | stable, well-calibrated, standard baseline | dominated by the huge sea/land areas, so rare classes (ship, oil) are under-learned |
| **Weighted CE** | CE with class weights (Section 3) | rare-class pixels count more | weights that are too large cause over-prediction of rare classes (false positives) and noisy training |
| **Dice loss** | 1 − soft Dice overlap per class, averaged over classes | directly optimises overlap; every class counts equally, whatever its size | gradients can be noisy for classes that are tiny or missing in a batch |
| **Focal loss** | CE × (1 − p_true)^γ, γ = 2 | down-weights easy pixels (open sea) and focuses on hard ones (oil vs look-alike borders) | can slow early training; γ is an extra hyper-parameter |

We combine a pixel-wise loss with Dice (CE + Dice, Focal + Dice). This is a common and robust choice for imbalanced segmentation.

In [ ]:
class DiceLoss(nn.Module):
    """Soft multi-class Dice loss: 1 - mean Dice over the classes."""
    def __init__(self, num_classes, smooth=1.0):
        super().__init__()
        self.num_classes, self.smooth = num_classes, smooth

    def forward(self, logits, target):
        probs = logits.softmax(dim=1)
        one_hot = F.one_hot(target, self.num_classes).permute(0, 3, 1, 2).float()
        dims = (0, 2, 3)
        intersection = (probs * one_hot).sum(dims)
        cardinality = probs.sum(dims) + one_hot.sum(dims)
        dice = (2 * intersection + self.smooth) / (cardinality + self.smooth)
        return 1 - dice.mean()


class FocalLoss(nn.Module):
    """Focal loss: cross-entropy scaled by (1 - p_true)^gamma."""
    def __init__(self, gamma=2.0):
        super().__init__()
        self.gamma = gamma

    def forward(self, logits, target):
        ce = F.cross_entropy(logits, target, reduction="none")
        p_true = torch.exp(-ce)
        return ((1 - p_true) ** self.gamma * ce).mean()


class ComboLoss(nn.Module):
    """Sum of two losses, e.g. weighted CE + Dice."""
    def __init__(self, first, second):
        super().__init__()
        self.first, self.second = first, second

    def forward(self, logits, target):
        return self.first(logits, target) + self.second(logits, target)


def make_loss(name):
    w = CLASS_WEIGHTS.to(DEVICE)
    if name == "ce":
        return nn.CrossEntropyLoss()
    if name == "wce_dice":
        return ComboLoss(nn.CrossEntropyLoss(weight=w), DiceLoss(NUM_CLASSES))
    if name == "focal_dice":
        return ComboLoss(FocalLoss(gamma=2.0), DiceLoss(NUM_CLASSES))
    raise ValueError(name)

# quick check that every loss works on random data
_logits, _target = torch.randn(2, NUM_CLASSES, 16, 16, device=DEVICE), torch.randint(0, NUM_CLASSES, (2, 16, 16), device=DEVICE)
print({name: round(make_loss(name)(_logits, _target).item(), 4) for name in ["ce", "wce_dice", "focal_dice"]})

## 7. Training loop, metrics and checkpoints

**Metrics** come from a 5×5 **confusion matrix** (rows = true class, columns = predicted class):
* IoU_c = TP / (TP + FP + FN), **mIoU** = mean IoU over the 5 classes (our main, class-sensitive metric)
* Dice_c (= F1) = 2TP / (2TP + FP + FN)
* Pixel accuracy = correct pixels / all pixels (supplementary only, because the sea dominates it)

**Checkpoint rule:** after every epoch we compute the validation mIoU. The epoch with the **highest
validation mIoU** is saved as `best.pth`. `last.pth` is saved every epoch, so an interrupted run can resume.
`history.csv` is also written every epoch.

Note: *train* metrics are measured on augmented batches while the model is learning. *Validation* metrics use the
model in eval mode with no augmentation, and are computed at the training resolution.

In [ ]:
def update_confusion(cm, pred, target):
    """Add a batch to the confusion matrix cm[true, pred]."""
    idx = target.reshape(-1) * NUM_CLASSES + pred.reshape(-1)
    cm += torch.bincount(idx, minlength=NUM_CLASSES ** 2).reshape(NUM_CLASSES, NUM_CLASSES)


def metrics_from_confusion(cm):
    cm = cm.double().cpu().numpy() if torch.is_tensor(cm) else np.asarray(cm, np.float64)
    tp = np.diag(cm)
    fp, fn = cm.sum(axis=0) - tp, cm.sum(axis=1) - tp
    with np.errstate(divide="ignore", invalid="ignore"):
        iou = tp / (tp + fp + fn)              # NaN if a class never appears and is never predicted
        dice = 2 * tp / (2 * tp + fp + fn)
    return {"miou": float(np.nanmean(iou)), "pixel_acc": float(tp.sum() / cm.sum()),
            "iou": iou.tolist(), "dice": dice.tolist()}


def run_epoch(model, loader, loss_fn, optimizer=None, scaler=None, freeze_encoder=False):
    """One pass over the data. Trains if an optimizer is given, otherwise only evaluates."""
    training = optimizer is not None
    model.train(training)
    if training and freeze_encoder:
        model.encoder.eval()        # frozen encoder: also keep its BatchNorm statistics fixed
    cm = torch.zeros(NUM_CLASSES, NUM_CLASSES, dtype=torch.long, device=DEVICE)
    total_loss, n_images = 0.0, 0
    for images, masks in loader:
        images, masks = images.to(DEVICE, non_blocking=True), masks.to(DEVICE, non_blocking=True)
        with torch.set_grad_enabled(training), torch.autocast(device_type=DEVICE, dtype=torch.float16, enabled=USE_AMP):
            logits = model(images)
        loss = loss_fn(logits.float(), masks)        # loss in float32 for numerical stability
        if training:
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        total_loss += loss.item() * images.size(0)
        n_images += images.size(0)
        update_confusion(cm, logits.argmax(dim=1), masks)
    return total_loss / n_images, metrics_from_confusion(cm)

In [ ]:
def build_model(exp):
    if exp["model"] == "unet":
        return UNet(in_channels=3, num_classes=NUM_CLASSES, base=UNET_BASE)
    return build_deeplab(pretrained=True)


def make_optimizer(model, exp):
    """AdamW. For DeepLabV3+ the pretrained encoder gets a smaller learning rate than the new decoder/head."""
    if exp["model"] == "deeplab":
        encoder_ids = {id(p) for p in model.encoder.parameters()}
        groups = [{"params": list(model.encoder.parameters()), "lr": exp["lr"] * exp["encoder_lr_mult"]},
                  {"params": [p for p in model.parameters() if id(p) not in encoder_ids], "lr": exp["lr"]}]
    else:
        groups = [{"params": list(model.parameters()), "lr": exp["lr"]}]
    return torch.optim.AdamW(groups, weight_decay=1e-4)


def set_encoder_trainable(model, trainable):
    for p in model.encoder.parameters():
        p.requires_grad = trainable


def atomic_save(obj, path):
    """Write to a temporary file first, then rename. A run killed mid-save can't leave a broken checkpoint."""
    tmp = Path(str(path) + ".tmp")
    torch.save(obj, tmp)
    os.replace(tmp, path)


def save_best(model, exp, path, epoch, val_metrics):
    """Save the weights plus everything needed to use the model later (backend / mobile app)."""
    mean, std = norm_stats(exp["model"])
    atomic_save({
        "model_type": exp["model"], "experiment": exp["name"], "state_dict": model.state_dict(),
        "num_classes": NUM_CLASSES, "class_names": CLASS_NAMES, "class_colors": CLASS_COLORS.tolist(),
        "img_size": [IMG_H, IMG_W], "mean": list(mean), "std": list(std),
        "unet_base": UNET_BASE, "encoder_name": ENCODER,
        "epoch": epoch, "val_miou": float(val_metrics["miou"]), "val_iou": [float(v) for v in val_metrics["iou"]],
    }, path)


def train_experiment(exp):
    exp_dir = OUT_DIR / exp["name"]
    exp_dir.mkdir(parents=True, exist_ok=True)
    if (exp_dir / "DONE").exists():                      # already finished (e.g. in an earlier session)
        print(f"[{exp['name']}] already trained, loading its history")
        return pd.read_csv(exp_dir / "history.csv")
    with open(exp_dir / "config.json", "w") as f:
        json.dump({**exp, "epochs": EPOCHS, "img_size": [IMG_H, IMG_W], "batch_size": BATCH_SIZE, "seed": SEED}, f, indent=1)

    seed_everything()
    mean, std = norm_stats(exp["model"])
    loader_gen = torch.Generator().manual_seed(SEED)
    train_loader = DataLoader(OilSpillDataset(train_images, train_masks, mean, std, augment=True),
                              batch_size=BATCH_SIZE, shuffle=True, drop_last=True, num_workers=NUM_WORKERS,
                              worker_init_fn=seed_worker, generator=loader_gen, pin_memory=DEVICE == "cuda")
    val_loader = DataLoader(OilSpillDataset(val_images, val_masks, mean, std, augment=False),
                            batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

    model = build_model(exp).to(DEVICE)
    loss_fn = make_loss(exp["loss"])
    optimizer = make_optimizer(model, exp)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

    history, best_miou, start_epoch = [], -1.0, 1
    last_path = exp_dir / "last.pth"
    if last_path.exists():                               # resume an interrupted run
        try:
            ck = torch.load(last_path, map_location=DEVICE, weights_only=False)
            model.load_state_dict(ck["model"]); optimizer.load_state_dict(ck["optimizer"])
            scheduler.load_state_dict(ck["scheduler"]); scaler.load_state_dict(ck["scaler"])
            history, best_miou, start_epoch = ck["history"], ck["best_miou"], ck["epoch"] + 1
            print(f"[{exp['name']}] resuming from epoch {start_epoch}")
        except Exception as e:
            print(f"[{exp['name']}] could not read last.pth ({e}), starting from scratch")
            model = build_model(exp).to(DEVICE)
            optimizer = make_optimizer(model, exp)
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

    for epoch in range(start_epoch, EPOCHS + 1):
        frozen = exp["model"] == "deeplab" and epoch <= exp["freeze_epochs"]
        if exp["model"] == "deeplab":
            set_encoder_trainable(model, not frozen)
        t0 = time.time()
        train_loss, train_m = run_epoch(model, train_loader, loss_fn, optimizer, scaler, freeze_encoder=frozen)
        with torch.no_grad():
            val_loss, val_m = run_epoch(model, val_loader, loss_fn)
        lr_now = optimizer.param_groups[-1]["lr"]
        scheduler.step()
        epoch_time = time.time() - t0

        row = {"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss,
               "train_miou": train_m["miou"], "val_miou": val_m["miou"],
               "train_pixel_acc": train_m["pixel_acc"], "val_pixel_acc": val_m["pixel_acc"],
               **{f"val_iou_{k}": v for k, v in zip(CLASS_KEYS, val_m["iou"])},
               "lr": lr_now, "encoder_frozen": frozen, "epoch_time_s": epoch_time}
        history.append(row)

        is_best = val_m["miou"] > best_miou
        if is_best:
            best_miou = val_m["miou"]
            save_best(model, exp, exp_dir / "best.pth", epoch, val_m)
        atomic_save({"model": model.state_dict(), "optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(),
                    "scaler": scaler.state_dict(), "epoch": epoch, "best_miou": best_miou, "history": history}, last_path)
        pd.DataFrame(history).to_csv(exp_dir / "history.csv", index=False)

        print(f"[{exp['name']}] ep {epoch:02d}/{EPOCHS} | train loss {train_loss:.4f} mIoU {train_m['miou']:.3f} | "
              f"val loss {val_loss:.4f} mIoU {val_m['miou']:.3f} (oil {val_m['iou'][OIL]:.3f}, ship {val_m['iou'][SHIP]:.3f}) | "
              f"{epoch_time:.0f}s{' | encoder frozen' if frozen else ''}{'  ⭐ best' if is_best else ''}")

    last_path.unlink()                                   # keep the output folder small
    (exp_dir / "DONE").write_text("finished")
    return pd.DataFrame(history)

## 8. Experiments

Each pair of experiments changes **one** thing, so the comparison stays fair. Everything else is fixed:
same split, seed, input size (320×640), batch size 8, 40 epochs, AdamW (weight decay 1e-4), cosine learning-rate
schedule, the same augmentations, and the same checkpoint rule (best validation mIoU).

| # | Experiment | Model | What changes | Loss | Fine-tuning |
|---|---|---|---|---|---|
| A1 | `unet_ce` | U-Net (scratch) | baseline | CE | – |
| A2 | `unet_wce_dice` | U-Net (scratch) | **loss** (vs A1) | weighted CE + Dice | – |
| B1 | `deeplab_frozen` | DeepLabV3+ R34 | baseline: **backbone frozen the whole run** | weighted CE + Dice | encoder never trained |
| B2 | `deeplab_finetune` | DeepLabV3+ R34 | **fine-tuning strategy** (vs B1) | weighted CE + Dice | frozen 5 epochs, then full fine-tuning (encoder LR × 0.1) |
| B3 | `deeplab_focal_dice` | DeepLabV3+ R34 | **loss** (vs B2) | Focal + Dice | same as B2 |

**Hypotheses.** A2 > A1 on the oil and ship IoU, because plain CE is dominated by sea pixels. B2 > B1,
because SAR images look very different from ImageNet photos, so the backbone has to adapt.
B3 vs B2 tests whether focusing on *hard* pixels (focal) works better than re-weighting *rare* classes (weighted CE).

In [ ]:
EXPERIMENTS = [
    dict(name="unet_ce", model="unet", loss="ce", lr=1e-3, freeze_epochs=0, encoder_lr_mult=1.0,
         description="U-Net, plain cross-entropy (baseline)"),
    dict(name="unet_wce_dice", model="unet", loss="wce_dice", lr=1e-3, freeze_epochs=0, encoder_lr_mult=1.0,
         description="U-Net, weighted CE + Dice (loss change)"),
    dict(name="deeplab_frozen", model="deeplab", loss="wce_dice", lr=1e-3, freeze_epochs=EPOCHS, encoder_lr_mult=0.1,
         description="DeepLabV3+, ImageNet backbone frozen for the whole run"),
    dict(name="deeplab_finetune", model="deeplab", loss="wce_dice", lr=1e-3, freeze_epochs=FREEZE_EPOCHS, encoder_lr_mult=0.1,
         description=f"DeepLabV3+, frozen {FREEZE_EPOCHS} epochs then full fine-tuning (encoder LR x0.1)"),
    dict(name="deeplab_focal_dice", model="deeplab", loss="focal_dice", lr=1e-3, freeze_epochs=FREEZE_EPOCHS, encoder_lr_mult=0.1,
         description="DeepLabV3+, fine-tuned, Focal + Dice (loss change)"),
]
display(pd.DataFrame(EXPERIMENTS))

In [ ]:
histories = {}
for exp in EXPERIMENTS:
    print("=" * 110, f"\n{exp['name']}: {exp['description']}")
    histories[exp["name"]] = train_experiment(exp)

## 9. Training curves and overfitting analysis
For each experiment we plot loss, mIoU (train vs validation) and the validation IoU of every class.
The dashed line marks the selected checkpoint (best validation mIoU). The text below each plot gives
**automatic hints** from the curves. Confirm them by looking at the plots before writing the discussion.

In [ ]:
def plot_history(name, h):
    best = h.loc[h.val_miou.idxmax()]
    fig, axes = plt.subplots(1, 3, figsize=(19, 4.3))
    axes[0].plot(h.epoch, h.train_loss, label="train"); axes[0].plot(h.epoch, h.val_loss, label="validation")
    axes[0].set(title=f"{name}: loss", xlabel="epoch", ylabel="loss")
    axes[1].plot(h.epoch, h.train_miou, label="train"); axes[1].plot(h.epoch, h.val_miou, label="validation")
    axes[1].set(title=f"{name}: mean IoU", xlabel="epoch", ylabel="mIoU")
    for c, key in enumerate(CLASS_KEYS):
        color = CLASS_COLORS[c] / 255 if c else (0.4, 0.4, 0.4)
        axes[2].plot(h.epoch, h[f"val_iou_{key}"], label=CLASS_NAMES[c], color=color)
    axes[2].set(title=f"{name}: validation IoU per class", xlabel="epoch", ylabel="IoU")
    for ax in axes:
        ax.axvline(best.epoch, ls="--", c="gray", lw=1)
        ax.legend(); ax.grid(alpha=0.3)
    if "encoder_frozen" in h and h.encoder_frozen.any() and not h.encoder_frozen.all():
        unfreeze = h.loc[~h.encoder_frozen, "epoch"].min()
        for ax in axes:
            ax.axvspan(h.epoch.min(), unfreeze - 0.5, color="lightblue", alpha=0.25)
    plt.tight_layout(); plt.savefig(OUT_DIR / name / "curves.png", dpi=110); plt.show()


def describe_curves(h):
    best = h.loc[h.val_miou.idxmax()]
    last = h.iloc[-1]
    min_loss_epoch = int(h.loc[h.val_loss.idxmin(), "epoch"])
    notes = [f"selected checkpoint: epoch {int(best.epoch)} (highest validation mIoU = {best.val_miou:.3f})",
             f"lowest validation loss at epoch {min_loss_epoch}; final train/val mIoU gap = {last.train_miou - last.val_miou:+.3f}",
             f"training time: {h.epoch_time_s.sum() / 60:.1f} min ({h.epoch_time_s.mean():.0f} s/epoch)"]
    tail = h.tail(max(3, len(h) // 5))
    if last.val_loss > 1.15 * h.val_loss.min() and last.train_loss < h.train_loss.iloc[min_loss_epoch - 1]:
        notes.append("hint: validation loss rises while training loss keeps falling -> signs of OVERFITTING "
                     f"after epoch ~{min_loss_epoch}")
    elif last.train_miou < 0.5 and abs(last.train_miou - last.val_miou) < 0.05:
        notes.append("hint: train and validation mIoU are both low and close -> possible UNDERFITTING")
    else:
        notes.append("hint: no strong overfitting signal; curves look like reasonable convergence")
    if tail.val_miou.std() > 0.03:
        notes.append(f"hint: validation mIoU fluctuates (std {tail.val_miou.std():.3f} over the last epochs) -> somewhat unstable")
    return notes


for exp in EXPERIMENTS:
    h = histories[exp["name"]]
    plot_history(exp["name"], h)
    print("\n".join("  • " + s for s in describe_curves(h)))

## 10. Final evaluation on the untouched test set
Each best checkpoint (picked on **validation** mIoU) is now evaluated **once** on the 110 test images.
Predictions are made at 320×640 and the logits are upsampled to the **original resolution**. All metrics are computed
against the original full-size masks.

In [ ]:
test_data = [(load_image(ip), load_mask(lp)) for ip, lp in test_pairs]   # full resolution
print(f"{len(test_data)} test images loaded, e.g. image {test_data[0][0].shape}, mask {test_data[0][1].shape}")


def load_trained(name):
    ck = torch.load(OUT_DIR / name / "best.pth", map_location=DEVICE, weights_only=False)
    model = UNet(num_classes=NUM_CLASSES, base=ck["unet_base"]) if ck["model_type"] == "unet" else build_deeplab(pretrained=False)
    model.load_state_dict(ck["state_dict"])
    return model.to(DEVICE).eval(), ck


@torch.no_grad()
def predict(model, image, mean, std):
    """Full-resolution class mask for one RGB image."""
    h, w = image.shape[:2]
    x = to_tensor(cv2.resize(image, (IMG_W, IMG_H), interpolation=cv2.INTER_AREA), mean, std)[None].to(DEVICE)
    logits = F.interpolate(model(x).float(), size=(h, w), mode="bilinear", align_corners=False)
    return logits.argmax(dim=1)[0].cpu().numpy().astype(np.uint8)


def evaluate_on_test(name):
    model, ck = load_trained(name)
    cm = torch.zeros(NUM_CLASSES, NUM_CLASSES, dtype=torch.long)
    per_image = []
    for (ip, _), (image, mask) in zip(test_pairs, test_data):
        pred = predict(model, image, ck["mean"], ck["std"])
        assert pred.shape == mask.shape
        update_confusion(cm, torch.from_numpy(pred).long(), torch.from_numpy(mask).long())
        per_image.append({"file": ip.name,
                          "oil_px": int((mask == OIL).sum()),
                          "oil_as_lookalike": int(((mask == OIL) & (pred == LOOKALIKE)).sum()),
                          "lookalike_as_oil": int(((mask == LOOKALIKE) & (pred == OIL)).sum()),
                          "ship_px": int((mask == SHIP).sum()),
                          "ship_hit": int(((mask == SHIP) & (pred == SHIP)).sum())})
    result = metrics_from_confusion(cm)
    result.update(confusion=cm.tolist(), best_epoch=ck["epoch"], val_miou=ck["val_miou"], per_image=per_image)
    with open(OUT_DIR / name / "test_metrics.json", "w") as f:
        json.dump(result, f, indent=1)
    del model
    return result


test_results = {}
for exp in EXPERIMENTS:
    t0 = time.time()
    test_results[exp["name"]] = evaluate_on_test(exp["name"])
    print(f"{exp['name']:<20} test mIoU {test_results[exp['name']]['miou']:.4f} ({time.time() - t0:.0f}s)")

In [ ]:
# Comparison table for all experiments
rows = []
for exp in EXPERIMENTS:
    r, h = test_results[exp["name"]], histories[exp["name"]]
    row = {"experiment": exp["name"], "model": "U-Net" if exp["model"] == "unet" else "DeepLabV3+",
           "loss": exp["loss"], "best_epoch": r["best_epoch"], "val_mIoU": r["val_miou"],
           "test_mIoU": r["miou"], "test_pixel_acc": r["pixel_acc"]}
    row.update({f"IoU_{k}": v for k, v in zip(CLASS_KEYS, r["iou"])})
    row.update({f"Dice_{k}": v for k, v in zip(CLASS_KEYS, r["dice"])})
    row["train_min"] = h.epoch_time_s.sum() / 60
    rows.append(row)
results = pd.DataFrame(rows)
results.to_csv(OUT_DIR / "results_table.csv", index=False)


def to_markdown(df, digits=3):
    """Simple Markdown table, ready to paste into the report."""
    cols = list(df.columns)
    lines = ["| " + " | ".join(cols) + " |", "|" + "---|" * len(cols)]
    for _, r in df.iterrows():
        lines.append("| " + " | ".join(f"{v:.{digits}f}" if isinstance(v, float) else str(v) for v in r) + " |")
    return "\n".join(lines)


(OUT_DIR / "results_table.md").write_text(to_markdown(results))
display(results.round(3))

In [ ]:
# Per-class IoU of every experiment
fig, ax = plt.subplots(figsize=(14, 4.5))
width = 0.8 / len(EXPERIMENTS)
for j, exp in enumerate(EXPERIMENTS):
    ax.bar(np.arange(NUM_CLASSES) + j * width, test_results[exp["name"]]["iou"], width, label=exp["name"])
ax.set_xticks(np.arange(NUM_CLASSES) + width * (len(EXPERIMENTS) - 1) / 2, CLASS_NAMES)
ax.set(ylabel="test IoU", title="Per-class test IoU of every experiment", ylim=(0, 1))
ax.legend(); ax.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.savefig(OUT_DIR / "per_class_iou.png", dpi=120); plt.show()

# Confusion matrices, normalised per true class (each row sums to 1 = recall distribution)
fig, axes = plt.subplots(1, len(EXPERIMENTS), figsize=(5 * len(EXPERIMENTS), 4.6))
short = ["sea", "oil", "look", "ship", "land"]
for ax, exp in zip(np.atleast_1d(axes), EXPERIMENTS):
    cm = np.array(test_results[exp["name"]]["confusion"], dtype=np.float64)
    cm_norm = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)
    ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
    for a in range(NUM_CLASSES):
        for b in range(NUM_CLASSES):
            ax.text(b, a, f"{cm_norm[a, b]:.2f}", ha="center", va="center", fontsize=8,
                    color="white" if cm_norm[a, b] > 0.5 else "black")
    ax.set_xticks(range(NUM_CLASSES), short); ax.set_yticks(range(NUM_CLASSES), short)
    ax.set(xlabel="predicted", ylabel="true", title=exp["name"])
plt.tight_layout(); plt.savefig(OUT_DIR / "confusion_matrices.png", dpi=120); plt.show()

**Reading the table.** Pixel accuracy is always high, because about 80–90% of all pixels are sea or land.
A model that never predicts a single ship can still score well above 90% pixel accuracy. That is why we
rank models by **mIoU** and look at the **oil-spill, look-alike and ship IoU/Dice**. The confusion matrices
show where errors go, mostly **oil ↔ look-alike** (both are dark, smooth patches in SAR) and **ship → sea** (ships are only a few pixels).

## 11. Qualitative results and failure cases
The best U-Net and the best DeepLabV3+ are picked by **validation** mIoU (not by test score).

In [ ]:
best_unet = max([e for e in EXPERIMENTS if e["model"] == "unet"], key=lambda e: test_results[e["name"]]["val_miou"])["name"]
best_deeplab = max([e for e in EXPERIMENTS if e["model"] == "deeplab"], key=lambda e: test_results[e["name"]]["val_miou"])["name"]
print("best U-Net (by val mIoU):", best_unet, "| best DeepLabV3+ (by val mIoU):", best_deeplab)
final_models = {name: load_trained(name) for name in [best_unet, best_deeplab]}

def predict_with(name, image):
    model, ck = final_models[name]
    return predict(model, image, ck["mean"], ck["std"])

def show_predictions(indices, title, names, filename):
    cols = 2 + len(names)
    fig, axes = plt.subplots(len(indices), cols, figsize=(4.6 * cols, 3.3 * len(indices)))
    axes = np.atleast_2d(axes)
    for row, i in enumerate(indices):
        image, mask = test_data[i]
        axes[row, 0].imshow(image); axes[row, 0].set_title(f"SAR image ({test_pairs[i][0].name})", fontsize=9)
        axes[row, 1].imshow(colorize(mask)); axes[row, 1].set_title("ground truth", fontsize=9)
        for k, name in enumerate(names):
            pred = predict_with(name, image)
            m = metrics_from_confusion(np.bincount(mask.ravel().astype(np.int64) * NUM_CLASSES + pred.ravel(),
                                                   minlength=NUM_CLASSES ** 2).reshape(NUM_CLASSES, NUM_CLASSES))
            axes[row, 2 + k].imshow(colorize(pred)); axes[row, 2 + k].set_title(f"{name} (image mIoU {m['miou']:.2f})", fontsize=9)
        for ax in axes[row]:
            ax.axis("off")
    fig.legend(handles=LEGEND, loc="lower center", ncol=5)
    fig.suptitle(title)
    plt.tight_layout(rect=(0, 0.04, 1, 0.97))
    plt.savefig(OUT_DIR / filename, dpi=110)
    plt.show()

# Representative examples: the test image with the most pixels of each important class
representative = []
for c in [OIL, LOOKALIKE, SHIP, 4]:
    counts = [int((m == c).sum()) for _, m in test_data]
    if max(counts) > 0:
        representative.append(int(np.argmax(counts)))
representative = list(dict.fromkeys(representative))
show_predictions(representative, "Qualitative results: input, ground truth, predictions", [best_unet, best_deeplab],
                 "qualitative_results.png")

In [ ]:
# Failure cases of the best DeepLabV3+ model
per_image = pd.DataFrame(test_results[best_deeplab]["per_image"])
per_image["oil_lookalike_confusion"] = per_image.oil_as_lookalike + per_image.lookalike_as_oil
per_image["ship_missed_ratio"] = 1 - per_image.ship_hit / per_image.ship_px.replace(0, np.nan)

confused = per_image.sort_values("oil_lookalike_confusion", ascending=False).head(3)
confused = confused[confused.oil_lookalike_confusion > 0]
display(confused[["file", "oil_px", "oil_as_lookalike", "lookalike_as_oil"]])
if len(confused):
    show_predictions(list(confused.index), "Failure cases: oil spill vs look-alike confusion", [best_unet, best_deeplab],
                     "failures_oil_vs_lookalike.png")

missed = per_image[per_image.ship_px > 0].sort_values("ship_missed_ratio", ascending=False).head(3)
display(missed[["file", "ship_px", "ship_hit", "ship_missed_ratio"]])
if len(missed):
    show_predictions(list(missed.index), "Failure cases: missed small ships", [best_unet, best_deeplab],
                     "failures_missed_ships.png")

## 12. Export for the mobile app
We copy the two selected checkpoints (with their test metrics) and a few test images into `outputs/export/`
and zip it. Unzip `export.zip` into the project's `backend/` folder:
`export/unet.pth` and `export/deeplab.pth` go to `backend/weights/`, and `export/samples/` goes to `backend/samples/`.

In [ ]:
EXPORT_DIR = OUT_DIR / "export"
(EXPORT_DIR / "weights").mkdir(parents=True, exist_ok=True)
for key, name in [("unet", best_unet), ("deeplab", best_deeplab)]:
    ck = torch.load(OUT_DIR / name / "best.pth", map_location="cpu", weights_only=False)
    r = test_results[name]
    ck["test_metrics"] = {"miou": r["miou"], "pixel_acc": r["pixel_acc"], "iou": r["iou"], "dice": r["dice"]}
    torch.save(ck, EXPORT_DIR / "weights" / f"{key}.pth")

for sub in ["images", "labels"]:
    (EXPORT_DIR / "samples" / sub).mkdir(parents=True, exist_ok=True)
for i in list(dict.fromkeys(representative + list(confused.index) + list(missed.index)))[:8]:
    ip, lp = test_pairs[i]
    shutil.copy(ip, EXPORT_DIR / "samples" / "images" / ip.name)
    shutil.copy(lp, EXPORT_DIR / "samples" / "labels" / lp.name)

shutil.copy(OUT_DIR / "results_table.csv", EXPORT_DIR / "results_table.csv")
with open(EXPORT_DIR / "class_mapping.json", "w") as f:
    json.dump({"class_names": CLASS_NAMES, "class_colors": CLASS_COLORS.tolist()}, f, indent=1)
shutil.make_archive(str(OUT_DIR / "export"), "zip", EXPORT_DIR)
print("export ready:", OUT_DIR / "export.zip")
for p in sorted(EXPORT_DIR.rglob("*")):
    if p.is_file():
        print(f"  {p.relative_to(OUT_DIR)}  ({p.stat().st_size / 1e6:.1f} MB)")

## 13. Discussion (write your own after the run)
Use the numbers above. Don't copy any numbers that did not come from this run.
* **U-Net A1 vs A2 (loss):** did weighted CE + Dice raise the oil-spill / ship IoU? Did pixel accuracy or sea IoU drop?
* **DeepLabV3+ B1 vs B2 (fine-tuning):** how much did unfreezing the ImageNet backbone help on SAR data?
* **B2 vs B3 (loss):** focal + Dice vs weighted CE + Dice. Which rare class gained or lost?
* **U-Net vs DeepLabV3+:** which is better per class, and why (pretraining, ASPP multi-scale context, capacity)?
* **Overfitting:** use the curves and hints from Section 9, not a single final score.
* **Limitations:** input downscaled to 320×640 (tiny ships shrink to a few pixels); oil vs look-alike is ambiguous even for experts;
  GPU non-determinism; a single random split; limited epochs because of the Kaggle time limit.